# 02 — Baseline Model: Fine-tune YOLOv8n

**Mục tiêu:** Xây **mô hình cơ sở (A0)** cho bài phát hiện + nhận dạng mệnh giá tiền VN, làm mốc so sánh cho mọi cải tiến sau này.

**Đầu vào:** Dataset YOLO đã EDA (`Vietnamese Currency.v1i.yolov8`)  
**Đầu ra:** Checkpoint `best.pt`, metric val (mAP / P / R), ảnh inference mẫu, bảng số liệu cho báo cáo.

**Protocol cố định (để so sánh công bằng):**
- Cùng split Roboflow: `train` / `valid` / `test`
- Input size `640`
- Chọn checkpoint theo **val mAP50** (không dùng test để chọn model)
- Test chỉ mở **một lần** khi đã khóa cấu hình (ở notebook cải tiến / so sánh)

**Pipeline notebook:**
1. Kiểm tra môi trường (GPU / thư viện)
2. Chuẩn hóa nhãn polygon → bbox (giữ bản gốc)
3. Tạo `data.yaml` đường dẫn đúng
4. Train YOLOv8n baseline
5. Đánh giá trên validation + quan sát lỗi
6. Xuất metric / ghi nhận bước tiếp theo

## 0. Cài đặt thư viện (chạy 1 lần nếu thiếu)

In [1]:
# Bỏ comment nếu môi trường chưa có ultralytics / torch
# %pip install -q ultralytics pyyaml pandas matplotlib seaborn pillow opencv-python-headless

import importlib.util

needed = ["ultralytics", "yaml", "pandas", "matplotlib", "seaborn", "PIL", "torch"]
missing = [m for m in needed if importlib.util.find_spec(m if m != "PIL" else "PIL") is None]
print("Thiếu thư viện:" , missing if missing else "(không)")
if missing:
    print("→ Hãy bỏ comment dòng %pip install ở trên rồi chạy lại cell này.")

Thiếu thư viện: ['ultralytics', 'torch']
→ Hãy bỏ comment dòng %pip install ở trên rồi chạy lại cell này.


## 1. Import & kiểm tra thiết bị

In [2]:
from __future__ import annotations

from pathlib import Path
import json
import os
import platform
import random
import shutil
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from PIL import Image, ImageDraw

import torch
from ultralytics import YOLO

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Python   :", platform.python_version())
print("Torch    :", torch.__version__)
print("CUDA avail:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU      :", torch.cuda.get_device_name(0))
    print("VRAM     :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
print("Device dùng cho train:", DEVICE)

ModuleNotFoundError: No module named 'torch'

## 2. Cấu hình đường dẫn & siêu tham số baseline

Baseline A0 giữ setting **tối giản / mặc định hợp lý**: model nhỏ (`yolov8n`), augment mặc định của Ultralytics, chưa sweep threshold, chưa đổi kiến trúc. Mọi thay đổi sau này phải so với mốc này trên cùng protocol.

In [ ]:
# Đường dẫn dự án
NOTEBOOK_DIR = Path(".").resolve()
PROJECT_DIR = NOTEBOOK_DIR.parent  # nhan_dang_tien_Viet_Nam/
RAW_DATA_DIR = PROJECT_DIR / "Vietnamese Currency.v1i.yolov8"

# Dataset đã chuẩn hóa bbox (không ghi đè bản gốc Roboflow)
CLEAN_DATA_DIR = PROJECT_DIR / "datasets" / "vn_currency_yolo_bbox"
CLEAN_YAML = CLEAN_DATA_DIR / "data.yaml"

# Thư mục lưu chạy train
RUNS_DIR = PROJECT_DIR / "runs" / "detect"
EXPORT_DIR = NOTEBOOK_DIR / "baseline_exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

assert RAW_DATA_DIR.exists(), f"Không thấy dataset gốc: {RAW_DATA_DIR}"

# ===== Siêu tham số baseline (A0) =====
QUICK_RUN = False   # True: train ngắn để kiểm tra pipeline (vd 3 epoch)
MODEL_NAME = "yolov8n.pt"
IMGSZ = 640
EPOCHS = 3 if QUICK_RUN else 50
BATCH = 16 if torch.cuda.is_available() else 4
WORKERS = 4
PATIENCE = 20          # early stopping theo val
RUN_NAME = "vn_currency_yolov8n_baseline"

print("RAW_DATA_DIR :", RAW_DATA_DIR)
print("CLEAN_DATA_DIR:", CLEAN_DATA_DIR)
print("MODEL / EPOCHS / BATCH / IMGSZ:", MODEL_NAME, EPOCHS, BATCH, IMGSZ)
print("QUICK_RUN:", QUICK_RUN)

## 3. Chuẩn hóa nhãn: polygon → YOLO bbox

Từ EDA: ~70% nhãn đã là bbox, ~30% là polygon. Object detection cần **một định dạng thống nhất** `class xc yc w h` (chuẩn hóa 0–1).

Chiến lược:
- **Giữ nguyên** dataset Roboflow gốc
- Tạo bản sạch: **symlink ảnh** + **copy labels đã convert**
- Ghi `data.yaml` với đường dẫn tuyệt đối (tránh lỗi `../train/images` của bản export)

In [ ]:
SPLITS = ["train", "valid", "test"]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def line_to_bbox(line: str) -> str:
    """Chuyển 1 dòng YOLO (bbox hoặc polygon) → bbox thuần."""
    parts = line.strip().split()
    if len(parts) < 5:
        raise ValueError(f"Nhãn không hợp lệ: {line!r}")

    class_id = int(float(parts[0]))
    coords = [float(x) for x in parts[1:]]

    if len(coords) == 4:
        xc, yc, w, h = coords
    else:
        if len(coords) % 2 != 0:
            raise ValueError(f"Polygon lẻ tọa độ: {line!r}")
        xs, ys = coords[0::2], coords[1::2]
        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        w = max(x_max - x_min, 1e-6)
        h = max(y_max - y_min, 1e-6)
        xc = (x_min + x_max) / 2.0
        yc = (y_min + y_max) / 2.0

    # Clip nhẹ về [0,1] để tránh lỗi số học biên
    xc = float(np.clip(xc, 0.0, 1.0))
    yc = float(np.clip(yc, 0.0, 1.0))
    w = float(np.clip(w, 1e-6, 1.0))
    h = float(np.clip(h, 1e-6, 1.0))
    return f"{class_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}"


def convert_label_file(src: Path, dst: Path) -> dict:
    stats = {"n_lines": 0, "n_bbox": 0, "n_polygon": 0}
    out_lines = []
    for line in src.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        n_coord = len(line.split()) - 1
        if n_coord == 4:
            stats["n_bbox"] += 1
        else:
            stats["n_polygon"] += 1
        out_lines.append(line_to_bbox(line))
        stats["n_lines"] += 1
    dst.write_text("\n".join(out_lines) + ("\n" if out_lines else ""), encoding="utf-8")
    return stats


def ensure_link_or_copy(src: Path, dst: Path):
    """Ưu tiên symlink ảnh để tiết kiệm ổ đĩa; fallback copy nếu OS không cho."""
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    try:
        dst.symlink_to(src.resolve())
    except OSError:
        shutil.copy2(src, dst)


def build_clean_dataset(raw_dir: Path, clean_dir: Path, force: bool = False) -> pd.DataFrame:
    marker = clean_dir / ".converted_ok"
    if clean_dir.exists() and marker.exists() and not force:
        print("Đã có dataset sạch — bỏ qua convert. (force=True để tạo lại)")
        return pd.read_csv(clean_dir / "convert_stats.csv") if (clean_dir / "convert_stats.csv").exists() else pd.DataFrame()

    if clean_dir.exists() and force:
        shutil.rmtree(clean_dir)

    rows = []
    for split in SPLITS:
        src_img = raw_dir / split / "images"
        src_lbl = raw_dir / split / "labels"
        dst_img = clean_dir / split / "images"
        dst_lbl = clean_dir / split / "labels"
        dst_img.mkdir(parents=True, exist_ok=True)
        dst_lbl.mkdir(parents=True, exist_ok=True)

        images = [p for p in sorted(src_img.iterdir()) if p.suffix.lower() in IMAGE_EXTS]
        for img_path in images:
            ensure_link_or_copy(img_path, dst_img / img_path.name)
            lbl_src = src_lbl / f"{img_path.stem}.txt"
            lbl_dst = dst_lbl / f"{img_path.stem}.txt"
            if lbl_src.exists():
                st = convert_label_file(lbl_src, lbl_dst)
            else:
                lbl_dst.write_text("", encoding="utf-8")
                st = {"n_lines": 0, "n_bbox": 0, "n_polygon": 0}
            rows.append({"split": split, "image": img_path.name, **st})

    stats_df = pd.DataFrame(rows)
    stats_df.to_csv(clean_dir / "convert_stats.csv", index=False)
    marker.write_text(datetime.now().isoformat(), encoding="utf-8")
    return stats_df


convert_stats = build_clean_dataset(RAW_DATA_DIR, CLEAN_DATA_DIR, force=False)
if len(convert_stats):
    summary = convert_stats.groupby("split", as_index=False).agg(
        n_images=("image", "count"),
        n_objects=("n_lines", "sum"),
        n_bbox_src=("n_bbox", "sum"),
        n_polygon_src=("n_polygon", "sum"),
    )
    display(summary)
    print(
        f"Tổng object: {int(convert_stats['n_lines'].sum())} | "
        f"từ bbox={int(convert_stats['n_bbox'].sum())}, "
        f"từ polygon={int(convert_stats['n_polygon'].sum())}"
    )
print("CLEAN_DATA_DIR:", CLEAN_DATA_DIR)

## 4. Tạo `data.yaml` cho Ultralytics

In [ ]:
with open(RAW_DATA_DIR / "data.yaml", "r", encoding="utf-8") as f:
    raw_cfg = yaml.safe_load(f)

CLASS_NAMES = list(raw_cfg["names"])
NC = int(raw_cfg.get("nc", len(CLASS_NAMES)))

data_yaml = {
    "path": str(CLEAN_DATA_DIR.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": NC,
    "names": CLASS_NAMES,
}

with open(CLEAN_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False, allow_unicode=True)

print("Đã ghi:", CLEAN_YAML)
print(json.dumps(data_yaml, indent=2, ensure_ascii=False))
print("\nPhạm vi lớp baseline:", CLASS_NAMES)
print("(9 lớp — khớp dữ liệu hiện có; thiếu 500đ so với README đề tài)")

## 5. QA nhanh trước khi train

Kiểm tra: số ảnh/nhãn khớp, mọi dòng nhãn đúng 5 trường, class_id ∈ [0, nc).

In [ ]:
def qa_clean_dataset(data_dir: Path, class_names: list[str]) -> pd.DataFrame:
    nc = len(class_names)
    issues = []
    class_counts = {name: 0 for name in class_names}

    for split in SPLITS:
        img_dir = data_dir / split / "images"
        lbl_dir = data_dir / split / "labels"
        images = [p for p in img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS]
        labels = list(lbl_dir.glob("*.txt"))
        img_stems = {p.stem for p in images}
        lbl_stems = {p.stem for p in labels}

        if img_stems - lbl_stems:
            issues.append({"split": split, "issue": "ảnh thiếu label", "count": len(img_stems - lbl_stems)})
        if lbl_stems - img_stems:
            issues.append({"split": split, "issue": "label thiếu ảnh", "count": len(lbl_stems - img_stems)})

        bad_fmt = 0
        bad_id = 0
        for lp in labels:
            for line in lp.read_text(encoding="utf-8").splitlines():
                if not line.strip():
                    continue
                parts = line.split()
                if len(parts) != 5:
                    bad_fmt += 1
                    continue
                cid = int(float(parts[0]))
                if cid < 0 or cid >= nc:
                    bad_id += 1
                else:
                    class_counts[class_names[cid]] += 1
        if bad_fmt:
            issues.append({"split": split, "issue": "dòng ≠ 5 trường", "count": bad_fmt})
        if bad_id:
            issues.append({"split": split, "issue": "class_id ngoài phạm vi", "count": bad_id})

        print(f"{split:5s}: images={len(images):4d} | labels={len(labels):4d}")

    issues_df = pd.DataFrame(issues) if issues else pd.DataFrame(columns=["split", "issue", "count"])
    counts_df = (
        pd.DataFrame({"class_name": list(class_counts.keys()), "n_boxes": list(class_counts.values())})
        .assign(class_name=lambda d: pd.Categorical(d["class_name"], categories=sorted(class_names, key=int), ordered=True))
        .sort_values("class_name")
        .reset_index(drop=True)
    )
    return issues_df, counts_df


issues_df, counts_df = qa_clean_dataset(CLEAN_DATA_DIR, CLASS_NAMES)
print("\nQA issues:")
display(issues_df if len(issues_df) else pd.DataFrame({"status": ["OK — không phát hiện lỗi kỹ thuật"]}))
print("Phân bố class sau convert:")
display(counts_df)

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=counts_df, x="class_name", y="n_boxes", ax=ax, color="#4C78A8")
ax.set_title("Số box / class trên dataset sạch (train+valid+test)")
ax.set_xlabel("Mệnh giá")
ax.set_ylabel("Số box")
plt.tight_layout()
plt.show()

## 6. Huấn luyện baseline YOLOv8n

Fine-tune từ trọng số pretrained COCO. Checkpoint tốt nhất được chọn theo **fitness trên validation** (Ultralytics mặc định gắn với mAP).

> Nếu VRAM báo OOM: giảm `BATCH` xuống 8 hoặc 4 rồi chạy lại cell train.

In [ ]:
model = YOLO(MODEL_NAME)

train_results = model.train(
    data=str(CLEAN_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    optimizer="auto",
    seed=SEED,
    deterministic=True,
    patience=PATIENCE,
    save=True,
    plots=True,
    verbose=True,
    # Baseline: giữ augment mặc định của Ultralytics (không tinh chỉnh tay ở A0)
)

RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"
print("RUN_DIR :", RUN_DIR)
print("BEST_PT :", BEST_PT)
print("Exists  :", BEST_PT.exists())

## 7. Đọc đường cong học & metric training log

In [ ]:
# Nếu chạy lại notebook mà bỏ qua train: tự tìm run gần nhất
if "RUN_DIR" not in globals() or not Path(RUN_DIR).exists():
    cand = sorted((RUNS_DIR).glob(f"{RUN_NAME}*"), key=lambda p: p.stat().st_mtime)
    assert cand, f"Không tìm thấy run train trong {RUNS_DIR}"
    RUN_DIR = cand[-1]
    BEST_PT = RUN_DIR / "weights" / "best.pt"

results_csv = RUN_DIR / "results.csv"
assert results_csv.exists(), f"Thiếu results.csv tại {RUN_DIR}"

hist = pd.read_csv(results_csv)
hist.columns = [c.strip() for c in hist.columns]
display(hist.tail())

# Một số cột phổ biến của Ultralytics
epoch_col = "epoch" if "epoch" in hist.columns else hist.columns[0]
map50_col = next((c for c in hist.columns if "mAP50(B)" in c or c.endswith("mAP50") and "95" not in c), None)
map5095_col = next((c for c in hist.columns if "mAP50-95" in c), None)
prec_col = next((c for c in hist.columns if c.endswith("precision") or "precision" in c.lower()), None)
rec_col = next((c for c in hist.columns if c.endswith("recall") or c.lower().endswith("/recall")), None)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if map50_col:
    axes[0].plot(hist[epoch_col], hist[map50_col], label="mAP50")
if map5095_col:
    axes[0].plot(hist[epoch_col], hist[map5095_col], label="mAP50-95")
axes[0].set_title("Validation mAP theo epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("mAP")
axes[0].legend()

if prec_col and rec_col:
    axes[1].plot(hist[epoch_col], hist[prec_col], label="Precision")
    axes[1].plot(hist[epoch_col], hist[rec_col], label="Recall")
    axes[1].set_title("Validation Precision / Recall")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, "Không tìm thấy cột P/R", ha="center")
    axes[1].axis("off")

plt.tight_layout()
plt.show()

# Hiển thị sẵn plots do Ultralytics xuất (nếu có)
for plot_name in ["results.png", "confusion_matrix.png", "BoxPR_curve.png"]:
    p = RUN_DIR / plot_name
    if p.exists():
        img = Image.open(p)
        plt.figure(figsize=(10, 6))
        plt.imshow(img)
        plt.axis("off")
        plt.title(plot_name)
        plt.show()

## 8. Đánh giá chính thức trên **validation**

Dùng `best.pt`. Theo protocol: **val** để báo cáo baseline và chọn hướng cải tiến; **test** để dành cho lần đánh giá khóa cuối (notebook so sánh).

In [ ]:
best_model = YOLO(str(BEST_PT))

val_metrics = best_model.val(
    data=str(CLEAN_YAML),
    split="val",
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    plots=True,
    verbose=True,
)

def extract_det_metrics(metrics) -> dict:
    box = metrics.box
    out = {
        "mAP50": float(box.map50),
        "mAP50-95": float(box.map),
        "precision": float(box.mp),
        "recall": float(box.mr),
    }
    # Per-class AP50 nếu có
    if getattr(box, "ap50", None) is not None:
        ap50 = np.array(box.ap50).reshape(-1)
        for i, name in enumerate(CLASS_NAMES):
            if i < len(ap50):
                out[f"AP50_{name}"] = float(ap50[i])
    return out


baseline_val = extract_det_metrics(val_metrics)
baseline_val_df = pd.DataFrame(
    [{"metric": k, "value": v} for k, v in baseline_val.items()]
)
display(baseline_val_df)

print(
    f"Baseline VAL | mAP50={baseline_val['mAP50']:.4f} | "
    f"mAP50-95={baseline_val['mAP50-95']:.4f} | "
    f"P={baseline_val['precision']:.4f} | R={baseline_val['recall']:.4f}"
)

### 8.1. (Tuỳ chọn) Đo nhanh trên test — chỉ để tham khảo, không chọn model theo số này

In [ ]:
RUN_TEST_NOW = False  # Để True chỉ khi muốn xem trước; số test "chính thức" nên lấy ở bước so sánh cuối

baseline_test = None
if RUN_TEST_NOW:
    test_metrics = best_model.val(
        data=str(CLEAN_YAML),
        split="test",
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        plots=False,
        verbose=True,
    )
    baseline_test = extract_det_metrics(test_metrics)
    print("TEST (tham khảo):", baseline_test)
else:
    print("Đã bỏ qua test — đúng tinh thần khóa cấu hình trước, đánh giá test sau.")

## 9. Inference trực quan trên ảnh validation

Quan sát qualitative: model bắt được tờ tiền? nhầm mệnh giá nào? có hallucinate trên letterbox trắng không?

In [ ]:
val_img_dir = CLEAN_DATA_DIR / "valid" / "images"
val_images = sorted([p for p in val_img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS])
sample_imgs = random.sample(val_images, k=min(8, len(val_images)))

pred_dir = EXPORT_DIR / "val_preview"
pred_dir.mkdir(parents=True, exist_ok=True)

pred_results = best_model.predict(
    source=[str(p) for p in sample_imgs],
    imgsz=IMGSZ,
    conf=0.25,   # ngưỡng tạm; tối ưu theo cost sẽ làm ở bước cải tiến quyết định
    device=DEVICE,
    save=False,
    verbose=False,
)

cols = 4
rows = int(np.ceil(len(pred_results) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
axes = np.array(axes).reshape(-1)

for ax, r in zip(axes, pred_results):
    plotted = r.plot()  # BGR ndarray
    ax.imshow(plotted[:, :, ::-1])
    ax.set_title(Path(r.path).name[:28], fontsize=8)
    ax.axis("off")

for ax in axes[len(pred_results):]:
    ax.axis("off")

plt.suptitle("Baseline predictions trên ảnh valid (conf=0.25)", y=1.01)
plt.tight_layout()
plt.savefig(pred_dir / "grid_preview.png", dpi=150, bbox_inches="tight")
plt.show()
print("Đã lưu grid:", pred_dir / "grid_preview.png")

### 9.1. Ưu tiên xem ảnh nhiều tờ (khó hơn)

In [ ]:
def count_objects(label_path: Path) -> int:
    if not label_path.exists():
        return 0
    return sum(1 for ln in label_path.read_text(encoding="utf-8").splitlines() if ln.strip())


hard_imgs = []
for img in val_images:
    n = count_objects(CLEAN_DATA_DIR / "valid" / "labels" / f"{img.stem}.txt")
    if n >= 3:
        hard_imgs.append((n, img))
hard_imgs = [p for _, p in sorted(hard_imgs, key=lambda x: -x[0])[:8]]

if not hard_imgs:
    print("Không có ảnh valid ≥ 3 object — dùng mẫu ngẫu nhiên.")
    hard_imgs = sample_imgs

hard_preds = best_model.predict(
    source=[str(p) for p in hard_imgs],
    imgsz=IMGSZ,
    conf=0.25,
    device=DEVICE,
    verbose=False,
)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()
for ax, r in zip(axes, hard_preds):
    ax.imshow(r.plot()[:, :, ::-1])
    ax.set_title(Path(r.path).name[:28], fontsize=8)
    ax.axis("off")
for ax in axes[len(hard_preds):]:
    ax.axis("off")
plt.suptitle("Ảnh nhiều tờ — nơi baseline thường lộ điểm yếu", y=1.01)
plt.tight_layout()
plt.savefig(pred_dir / "hard_cases.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Xuất bảng metric baseline cho báo cáo

In [ ]:
meta = {
    "model": MODEL_NAME,
    "run_name": RUN_NAME,
    "run_dir": str(RUN_DIR),
    "best_pt": str(BEST_PT),
    "data_yaml": str(CLEAN_YAML),
    "imgsz": IMGSZ,
    "epochs_requested": EPOCHS,
    "batch": BATCH,
    "seed": SEED,
    "device": str(DEVICE),
    "n_classes": NC,
    "class_names": CLASS_NAMES,
    "quick_run": QUICK_RUN,
    "created_at": datetime.now().isoformat(timespec="seconds"),
}

metrics_row = {
    "model": "YOLOv8n baseline (A0)",
    "selected_by": "val mAP50 / Ultralytics fitness",
    "val_mAP50": baseline_val.get("mAP50"),
    "val_mAP50-95": baseline_val.get("mAP50-95"),
    "val_precision": baseline_val.get("precision"),
    "val_recall": baseline_val.get("recall"),
}
if baseline_test is not None:
    metrics_row.update({
        "test_mAP50": baseline_test.get("mAP50"),
        "test_mAP50-95": baseline_test.get("mAP50-95"),
        "test_precision": baseline_test.get("precision"),
        "test_recall": baseline_test.get("recall"),
    })

metrics_table = pd.DataFrame([metrics_row])
per_class = (
    pd.DataFrame(
        [
            {"class_name": name, "AP50_val": baseline_val.get(f"AP50_{name}")}
            for name in CLASS_NAMES
        ]
    )
    .assign(class_name=lambda d: pd.Categorical(d["class_name"], categories=sorted(CLASS_NAMES, key=int), ordered=True))
    .sort_values("class_name")
    .reset_index(drop=True)
)

metrics_table.to_csv(EXPORT_DIR / "baseline_val_metrics.csv", index=False)
per_class.to_csv(EXPORT_DIR / "baseline_val_AP50_per_class.csv", index=False)
with open(EXPORT_DIR / "baseline_run_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

# Copy best.pt ra chỗ dễ tìm
export_weight = EXPORT_DIR / "yolov8n_baseline_best.pt"
if BEST_PT.exists():
    shutil.copy2(BEST_PT, export_weight)

display(metrics_table)
display(per_class)
print("Đã xuất:")
print(" -", EXPORT_DIR / "baseline_val_metrics.csv")
print(" -", EXPORT_DIR / "baseline_val_AP50_per_class.csv")
print(" -", EXPORT_DIR / "baseline_run_meta.json")
print(" -", export_weight)

## 11. Tóm tắt & bước tiếp theo

Baseline A0 chỉ là **mốc**. Các hướng cải tiến nên đổi **một nhóm yếu tố** mỗi lần:

| Hướng | Việc làm | Mục đích so với baseline |
|-------|----------|---------------------------|
| A1 — Dữ liệu | Augment mạnh hơn, rà outlier nhãn | Tăng generalization |
| A2 — Capacity | `yolov8s` / train lâu hơn | Tăng sức biểu diễn |
| A3 — Quyết định | Sweep `conf` trên **val** theo chi phí FP/FN | Tối ưu ngưỡng vận hành |

Notebook kế tiếp gợi ý: `04_Improve_and_Compare.ipynb`.

In [ ]:
print("===== BASELINE DONE =====")
print(f"Model     : {MODEL_NAME}")
print(f"Classes   : {NC} → {CLASS_NAMES}")
print(f"Data      : {CLEAN_YAML}")
print(f"Best ckpt : {BEST_PT}")
print(
    f"VAL       : mAP50={baseline_val['mAP50']:.4f} | "
    f"mAP50-95={baseline_val['mAP50-95']:.4f} | "
    f"P={baseline_val['precision']:.4f} | R={baseline_val['recall']:.4f}"
)
print("\nGợi ý viết báo cáo:")
print("1) Nêu rõ đây là mô hình cơ sở fine-tune YOLOv8n, protocol cố định split/imgsz.")
print("2) Nhắc đã chuẩn hóa polygon → bbox trước train (từ kết quả EDA).")
print("3) Báo cáo metric VAL; chưa dùng test để chọn cấu hình.")
print("4) Quan sát ảnh nhiều tờ / nhầm class gần nhau để định hướng cải tiến.")
if QUICK_RUN:
    print("\n⚠ QUICK_RUN=True — đây chỉ là chạy thử pipeline. Đặt False rồi train lại đủ epoch trước khi đưa số vào báo cáo.")